In [18]:
# 02_feature_engineering.ipynb

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# 让 notebook 找到 src 包
PROJECT_ROOT = Path.cwd().parent   # 假设 notebook 在 TFM_25/notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

from src.config import (
    RAW_DATA_PATH,
    REG_TARGET_TOTEXPY2_RAW,
    REG_TARGET_TOTEXPY2_LOG,
    REG_BASELINE_TOTEXPY1,
    CLASS_TARGET_HIGHCOST_Y2,
    CLASS_TARGET_ANY_ED_Y2,
    CLASS_TARGET_ANY_IP_Y2,
)

from src.data_io import load_raw_meps, save_processed_meps
from src.preprocessing import preprocess_meps
from src.features import add_features  # 这是你接下来要写的 feature 工厂函数


PROJECT_ROOT: /Users/wenxi/Desktop/TFM_25


In [3]:
# 1. 读原始数据
df_raw = load_raw_meps()
print("df_raw:", df_raw.shape)

df_raw: (8292, 2648)


In [4]:
# 2. 预处理（select core + filter YEARIND/ALL5RDS + 缺失码 + 金额负值 + drop PREVCOVR/MORECOVR）
df_pre = preprocess_meps(df_raw)
print("df_pre:", df_pre.shape)

df_pre: (7812, 101)


In [4]:
from src.config import PROCESSED_DATA_PATH
from src.data_io import save_processed_meps

# 如果你想把 df_pre 存起来（推荐 parquet）
save_processed_meps(df_pre, PROCESSED_DATA_PATH)  # 默认路径在 config 里
print("Saved preprocessed dataset to:", PROCESSED_DATA_PATH)


Saved preprocessed dataset to: /Users/wenxi/Desktop/TFM_25/data/meps_panel27_processed.parquet


In [5]:
# feature engineering pipline

df_feat = add_features(df_pre)

df_feat.head()


,DUID,PID,DUPERSID,PANEL,YEARIND,ALL5RDS,DIED,INST,MILITARY,ENTRSRVY,...,DIABDXY1_M18_BIN,MULTIMORBIDITY_Y1,MULTIMORBIDITY_GE2,LOG_TOTEXPY1,ANY_ED_Y1,ANY_IP_Y1,LOG_TOTEXPY2,HIGHCOST_Y2,ANY_ED_Y2,ANY_IP_Y2
0,2790002,101,2790002101,27,1,1,0,0,0,0,...,1,1,0,7.595890,0,0,6.472346,0,0,0
1,2790002,102,2790002102,27,1,1,0,0,0,0,...,0,1,0,0.000000,0,0,7.546974,0,0,0
2,2790004,101,2790004101,27,1,1,0,0,0,0,...,0,0,0,7.379632,0,0,6.894670,0,0,0
3,2790006,101,2790006101,27,1,1,0,0,0,0,...,1,3,1,7.374629,0,0,7.180070,0,0,0
4,2790006,102,2790006102,27,1,1,0,0,0,0,...,0,0,0,5.017280,0,0,0.000000,0,0,0


In [6]:
from pathlib import Path
from src.config import PROJECT_ROOT

out_path = PROJECT_ROOT / "data" / "df_feat.parquet"
df_feat.to_parquet(out_path, index=False)
print("Saved to:", out_path)


Saved to: /Users/wenxi/Desktop/TFM_25/data/df_feat.parquet


In [19]:
from src.config import (
    REG_TARGET_TOTEXPY2_RAW,
    REG_TARGET_TOTEXPY2_LOG,
    CLASS_TARGET_HIGHCOST_Y2,
    CLASS_TARGET_ANY_ED_Y2,
    CLASS_TARGET_ANY_IP_Y2,
)

cols = [
    REG_TARGET_TOTEXPY2_RAW,
    REG_TARGET_TOTEXPY2_LOG,
    CLASS_TARGET_HIGHCOST_Y2,
    CLASS_TARGET_ANY_ED_Y2,
    CLASS_TARGET_ANY_IP_Y2,
]
[c for c in cols if c in df_feat.columns]


['TOTEXPY2', 'LOG_TOTEXPY2', 'HIGHCOST_Y2', 'ANY_ED_Y2', 'ANY_IP_Y2']

In [8]:
df_feat[[c for c in cols if c in df_feat.columns]].describe(include="all")


,TOTEXPY2,LOG_TOTEXPY2,HIGHCOST_Y2,ANY_ED_Y2,ANY_IP_Y2
count,7812.000000,7812.000000,7812.000000,7812.000000,7812.000000
mean,8304.398874,6.669023,0.100102,0.142729,0.072581
std,22264.311550,3.227354,0.300156,0.349819,0.259464
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,272.750000,5.612214,0.000000,0.000000,0.000000
50%,1815.500000,7.504667,0.000000,0.000000,0.000000
75%,7268.250000,8.891408,0.000000,0.000000,0.000000
max,574675.000000,13.261562,1.000000,1.000000,1.000000


In [9]:
# class balance
for c in [CLASS_TARGET_HIGHCOST_Y2, CLASS_TARGET_ANY_ED_Y2, CLASS_TARGET_ANY_IP_Y2]:
    if c in df_feat.columns:
        print(c, df_feat[c].value_counts(dropna=False), "\n")


HIGHCOST_Y2 HIGHCOST_Y2
0    7030
1     782
Name: count, dtype: int64 

ANY_ED_Y2 ANY_ED_Y2
0    6697
1    1115
Name: count, dtype: int64 

ANY_IP_Y2 ANY_IP_Y2
0    7245
1     567
Name: count, dtype: int64 



In [10]:
import numpy as np
import pandas as pd

# 1) define columns you NEVER want as model predictors
ID_COLS = ["DUPERSID", "DUID", "PID", "PANEL", "VARSTR", "VARPSU"]
WEIGHT_COLS = ["LONGWT", "LSAQWT"]

# targets 
TARGET_COLS = ["TOTEXPY2", "LOG_TOTEXPY2", "HIGHCOST_Y2", "ANY_ED_Y2", "ANY_IP_Y2"]

EXCLUDE = set(ID_COLS + WEIGHT_COLS + TARGET_COLS)

# 2) candidate feature columns = everything else
feature_candidates = [c for c in df_feat.columns if c not in EXCLUDE]

# 3) categorical = object/category
cat_cols = df_feat[feature_candidates].select_dtypes(include=["object", "category"]).columns.tolist()

# 4) numeric = number types (int/float/bool)
num_cols = df_feat[feature_candidates].select_dtypes(include=[np.number]).columns.tolist()

print("n feature candidates:", len(feature_candidates))
print("n cat:", len(cat_cols))
print("n num:", len(num_cols))

cat_cols[:20], num_cols[:20]


n feature candidates: 128
n cat: 7
n num: 121


(['AGE_GROUP',
  'RACE_ETH',
  'REGIONY1_CAT',
  'EDU_GROUP',
  'POVCATY1_CAT',
  'FAMSIZE_Y1_GRP',
  'INS_TYPE_Y1'],
 ['YEARIND',
  'ALL5RDS',
  'DIED',
  'INST',
  'MILITARY',
  'ENTRSRVY',
  'LEFTUS',
  'OTHER',
  'AGEY1X',
  'AGEY2X',
  'AGELSTY1',
  'AGELSTY2',
  'SEX',
  'RACETHX',
  'HISPANX',
  'EDUCYR',
  'REGIONY1',
  'REGIONY2',
  'FAMINCY1',
  'FAMINCY2'])

In [11]:
print("Categorical columns:\n", cat_cols)
print("\nNumeric columns:\n", num_cols)


Categorical columns:
 ['AGE_GROUP', 'RACE_ETH', 'REGIONY1_CAT', 'EDU_GROUP', 'POVCATY1_CAT', 'FAMSIZE_Y1_GRP', 'INS_TYPE_Y1']

Numeric columns:
 ['YEARIND', 'ALL5RDS', 'DIED', 'INST', 'MILITARY', 'ENTRSRVY', 'LEFTUS', 'OTHER', 'AGEY1X', 'AGEY2X', 'AGELSTY1', 'AGELSTY2', 'SEX', 'RACETHX', 'HISPANX', 'EDUCYR', 'REGIONY1', 'REGIONY2', 'FAMINCY1', 'FAMINCY2', 'POVCATY1', 'POVCATY2', 'POVLEVY1', 'POVLEVY2', 'FAMSZEY1', 'FAMSZEY2', 'RUSIZEY1', 'RUSIZEY2', 'INSCOVY1', 'INSCOVY2', 'INSURCY1', 'INSURCY2', 'UNINSY1', 'UNINSY2', 'EVRWRKY1', 'EVRWRKY2', 'EMPST1', 'EMPST2', 'EMPST3', 'EMPST4', 'EMPST5', 'UNEMPY1X', 'UNEMPY2X', 'RTHLTH1', 'RTHLTH3', 'RTHLTH5', 'MNHLTH1', 'MNHLTH3', 'MNHLTH5', 'HIBPDXY1', 'HIBPDXY2', 'CHDDXY1', 'CHDDXY2', 'STRKDXY1', 'STRKDXY2', 'CHOLDXY1', 'CHOLDXY2', 'ASTHDXY1', 'ASTHDXY2', 'DIABDXY1_M18', 'DIABDXY2_M18', 'ERTOTY1', 'ERTOTY2', 'IPDISY1', 'IPDISY2', 'TOTEXPY1', 'TOTTCHY1', 'TOTTCHY2', 'TOTSLFY1', 'TOTSLFY2', 'TOTMCRY1', 'TOTMCRY2', 'TOTMCDY1', 'TOTMCDY2', 'TOTPRVY1'

In [35]:
df_feat['INS_TYPE_Y1'].value_counts()

INS_TYPE_Y1
<65 any private                3905
<65 public only                1547
65+ Medicare + private          759
65+ Medicare only               699
<65 uninsured                   554
65+ Medicare + other public     310
65+ other coverage               29
65+ uninsured                     9
Name: count, dtype: int64

In [12]:
cat_cols = [
    "RACE_ETH",
    "REGIONY1_CAT",
    "EDU_GROUP",
    "POVCATY1_CAT",
    "FAMSIZE_Y1_GRP",
    "INS_TYPE_Y1",
]


drop AGE_GROUP from modeling because you already have continuous AGE (keeping both is redundant). Use AGE_GROUP only for plots/interpretation.

In [13]:
df_feat["EMP_INFO_R12"].value_counts(dropna=False)


EMP_INFO_R12
1    6492
0    1320
Name: count, dtype: int64

In [14]:
df_feat["EMP_ATTACHED_ANY_R12"].isna().mean()
df_feat["EMP_ATTACHED_ANY_R12"].value_counts(dropna=False)


EMP_ATTACHED_ANY_R12
1.0    3979
0.0    2513
NaN    1320
Name: count, dtype: int64

In [15]:
df_feat["EMP_ATTACHED_ANY_R12_FILL0"].isna().mean()

np.float64(0.0)

In [16]:
#Numeric (use engineered columns, not raw)

num_cols = [
    # demographics / SES
    "AGE",
    "SEX_BIN",
    "LOG_FAMINCY1",
    "FAMSIZE_Y1",

    # insurance (coarse flags) — only if you are NOT using INS_TYPE_Y1
    # If you keep INS_TYPE_Y1, drop these three to avoid redundancy
    # "ANY_PRIVATE_Y1", "PUBLIC_ONLY_Y1", "UNINSURED_Y1",

    # employment
    "WORKED_Y1",
    "ANY_UNEMP_COMP_Y1",
    "LOG_UNEMP_COMP_Y1",
    "EMP_INFO_R12",
    "EMP_ATTACHED_ANY_R12_FILL0",  # model-friendly version

    # health status baseline
    "RTHLTH1_FAIRPOOR",
    "MNHLTH1_FAIRPOOR",

    # chronic conditions baseline
    "HIBPDXY1_BIN",
    "CHDDXY1_BIN",
    "STRKDXY1_BIN",
    "CHOLDXY1_BIN",
    "ASTHDXY1_BIN",
    "DIABDXY1_M18_BIN",
    # "MULTIMORBIDITY_Y1",   # optional (can remove if you keep all *_BIN)

    # baseline utilisation/cost
    "LOG_TOTEXPY1",
    "ANY_ED_Y1",
    "ANY_IP_Y1",
]


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ],
    remainder="drop",
)


Notes:

For categoricals, imputing with most_frequent is fine for baseline.

OneHotEncoder(handle_unknown="ignore") prevents test-time errors.